# ml4t-india — Full Live Integration Tour

Exercises **every feature** of the library with a real Kite Connect session.

**Prerequisites:** `pip install ml4t-india[all]` and an active Kite Connect subscription.

Credentials are entered interactively — nothing is stored to disk or git.

In [ ]:
import asyncio
import datetime as dt
import getpass
import threading

from ml4t.india.kite.auth import generate_session, login_url
from ml4t.india.kite.client import AsyncKiteClient, KiteClient
from ml4t.india.live.kite_broker import KiteBroker
from ml4t.india.live.kite_ticker_feed import KiteTickerFeed
from ml4t.india.live.postbacks import PostbackHandler
from ml4t.india.backtest.charges import ZerodhaChargesModel, Segment
from ml4t.india.backtest.lot_sizing import round_to_lot, floor_to_lot
from ml4t.india.calendar import nse_calendar
from ml4t.india.options import OptionChain, compute_greeks
from ml4t.india.workflows import ResearchPipeline, DeploymentPipeline
from ml4t.india.data.kite import KiteProvider
from ml4t.backtest.types import OrderSide, OrderType

## Section 1: Login & Auth

In [ ]:
print("Enter your Kite Connect credentials (not stored anywhere):")
api_key = getpass.getpass("API key: ").strip()
api_secret = getpass.getpass("API secret: ").strip()

print(f"\nOpen this URL to log in:\n  {login_url(api_key)}")
print("\nAfter redirect, copy the ?request_token=XXX value from the URL.")
request_token = getpass.getpass("Request token: ").strip()

record = generate_session(api_key, api_secret, request_token)
print(f"Logged in as: {record.user_id}")
print(f"Token expires: ~06:00 IST tomorrow | is_expired={record.is_expired()}")

In [ ]:
sync_client = KiteClient.from_api_key(api_key=api_key, access_token=record.access_token)
client = AsyncKiteClient(sync_client)

profile = await client.profile()
print(f"User: {profile.get('user_name')} | Email: {profile.get('email')} | Broker: {profile.get('broker')}")

## Section 2: Instruments

In [ ]:
nse_instruments = await client.instruments("NSE")
print(f"NSE instruments: {len(nse_instruments)}")

infy = next(i for i in nse_instruments if i.get("tradingsymbol") == "INFY")
infy_token = infy["instrument_token"]
print(f"INFY: token={infy_token}, lot_size={infy.get('lot_size', 1)}, exchange={infy.get('exchange')}")

## Section 3: Historical OHLCV

In [ ]:
end = dt.datetime.now(dt.UTC)
start = end - dt.timedelta(days=30)
candles = await client.historical_data(
    instrument_token=infy_token,
    from_date=start,
    to_date=end,
    interval="day",
)
print(f"INFY daily candles (30 days): {len(candles)} bars")
for c in candles[-3:]:
    print(f"  {c['date'].date()} O={c['open']} H={c['high']} L={c['low']} C={c['close']} V={c['volume']}")
assert len(candles) > 0, "Expected at least one candle"

## Section 4: NSE Calendar

In [ ]:
cal = nse_calendar()
today = dt.date.today()
print(f"Today ({today}) is_session_day: {cal.is_session_day(today)}")
next_sess = cal.next_session(today)
print(f"Next session: {next_sess}")
open_t, close_t = cal.session_bounds(next_sess)
print(f"Session bounds (IST): {open_t.time()} — {close_t.time()}")
assert cal is not None

## Section 5: Backtest + Charges

In [ ]:
model = ZerodhaChargesModel(default_segment=Segment.EQUITY_DELIVERY)
charges = model.calculate(asset="NSE:INFY", quantity=10, price=1800.0)
print(f"Charges on 10 x INFY @ Rs 1800 (delivery):")
print(f"  Brokerage: Rs {charges.brokerage:.2f}")
print(f"  STT:       Rs {charges.stt:.2f}")
print(f"  GST:       Rs {charges.gst:.2f}")
print(f"  Total:     Rs {charges.total:.2f}")
assert charges.total >= 0

## Section 6: Lot Sizing

In [ ]:
lot_size = 50
print(f"round_to_lot(76, 50) = {round_to_lot(76, 50)}")
print(f"floor_to_lot(76, 50) = {floor_to_lot(76, 50)}")
assert round_to_lot(76, 50) == 100
assert floor_to_lot(76, 50) == 50

## Section 7: Option Chain

In [ ]:
nifty_options = [
    i for i in nse_instruments
    if i.get("name") == "NIFTY" and i.get("instrument_type") in ("CE", "PE")
]
expiries = sorted({i["expiry"] for i in nifty_options if i.get("expiry")})
nearest_expiry = expiries[0]
print(f"Nearest NIFTY expiry: {nearest_expiry}")

chain = OptionChain.from_instruments(nifty_options, underlying="NIFTY", expiry=nearest_expiry)

nifty_ltp_data = await client.ltp(["NSE:NIFTY 50"])
spot = list(nifty_ltp_data.values())[0]["last_price"]
atm = chain.atm_strike(spot)
calls, puts = chain.around_atm(spot, count=3)
print(f"Spot={spot}  ATM={atm}  PCR={chain.put_call_ratio():.3f}  MaxPain={chain.max_pain()}")
print(f"Near-ATM calls: {[s.strike for s in calls]}")
print(f"Near-ATM puts:  {[s.strike for s in puts]}")

## Section 8: Greeks (Black-Scholes)

In [ ]:
tte = (nearest_expiry - dt.date.today()).days / 365

greeks = compute_greeks(
    flag="CE",
    spot=spot,
    strike=float(atm),
    time_to_expiry=tte,
    risk_free_rate=0.065,
    volatility=0.15,
)
print(f"ATM Call Greeks  spot={spot}  strike={atm}  tte={tte:.4f}")
print(f"  Delta={greeks.delta:.4f}  Gamma={greeks.gamma:.6f}  Theta={greeks.theta:.4f}  Vega={greeks.vega:.4f}")
assert greeks.delta is not None

## Section 9: Live Broker (KiteBroker)

In [ ]:
broker = KiteBroker(client=client)
await broker.connect()
assert await broker.is_connected_async(), "Broker failed to connect"

cash = await broker.get_cash_async()
account_value = await broker.get_account_value_async()
positions = await broker.get_positions_async()
print(f"Cash:          Rs {cash:,.2f}")
print(f"Account value: Rs {account_value:,.2f}")
print(f"Open positions: {len(positions)}")
for asset, pos in positions.items():
    print(f"  {asset}: qty={pos.quantity} @ Rs {pos.entry_price:.2f}")

In [ ]:
print("Placing Rs 1 LIMIT BUY for 1 INFY (far off-market, zero fill risk)...")
order = await broker.submit_order_async(
    asset="NSE:INFY",
    quantity=1,
    side=OrderSide.BUY,
    order_type=OrderType.LIMIT,
    limit_price=1.0,
    product="CNC",
)
print(f"Order placed: id={order.order_id}  status={order.status}")

pending = await broker.get_pending_orders_async()
assert any(o.order_id == order.order_id for o in pending), "Order not in pending list"
print(f"Confirmed in pending orders ({len(pending)} total)")

cancelled = await broker.cancel_order_async(order.order_id)
assert cancelled is True
print(f"Order cancelled: {cancelled}")

await broker.disconnect()
print("KiteBroker disconnected")

## Section 10: Ticker Feed (KiteTickerFeed)

In [ ]:
ticks_received: list[dict] = []
connect_event = threading.Event()

feed = KiteTickerFeed(api_key=api_key, access_token=record.access_token, default_mode="ltp")

def _on_connect() -> None:
    feed.subscribe([infy_token], mode="ltp")
    connect_event.set()

def _on_ticks(ticks: list[dict]) -> None:
    ticks_received.extend(ticks)

feed.on_connect(_on_connect)
feed.on_ticks(_on_ticks)

await feed.start()
connected = connect_event.wait(timeout=15)

if connected and len(ticks_received) == 0:
    await asyncio.sleep(3)

feed.stop()
print(f"KiteTickerFeed: connected={connected}  ticks_received={len(ticks_received)}")
if ticks_received:
    print(f"  Sample tick: {ticks_received[0]}")
else:
    print("  No ticks (market closed or outside trading hours — feed connectivity verified)")

## Section 11: Research Pipeline

In [ ]:
provider = KiteProvider(client=sync_client)
research = ResearchPipeline(provider=provider, initial_cash=1_000_000)
print(f"ResearchPipeline: {research}")
print("Full run requires a Strategy subclass — see notebooks/04-backtest-with-charges.ipynb")
assert research is not None

## Section 12: Deployment Pipeline

In [ ]:
postbacks = PostbackHandler(api_secret=api_secret)
deployment = DeploymentPipeline(
    broker=broker,
    feed=feed,
    postbacks=postbacks,
    instrument_tokens=[infy_token],
)
print(f"DeploymentPipeline: {deployment}")
print("Full start requires a live Strategy — see notebooks/10-deployment-pipeline.ipynb")
assert deployment is not None

## Summary

In [ ]:
results = [
    ("1. Login & Auth",          record is not None and not record.is_expired()),
    ("2. Instruments",           len(nse_instruments) > 0),
    ("3. Historical OHLCV",      len(candles) > 0),
    ("4. NSE Calendar",          cal is not None),
    ("5. Backtest + Charges",    charges.total >= 0),
    ("6. Lot Sizing",            round_to_lot(76, 50) == 100),
    ("7. Option Chain",          atm is not None),
    ("8. Greeks",                greeks.delta is not None),
    ("9. Live Broker",           True),
    ("10. Ticker Feed",          connected),
    ("11. Research Pipeline",    research is not None),
    ("12. Deployment Pipeline",  deployment is not None),
]
print(f"{'Section':<32} Status")
print("-" * 42)
for name, ok in results:
    print(f"{name:<32} {'PASS' if ok else 'FAIL'}")
all_passed = all(ok for _, ok in results)
print(f"\nOverall: {'ALL PASS' if all_passed else 'SOME SECTIONS FAILED'}")